# Aperture photometery function

26 January 2026 | Klara Kurucova

This code can be used to conduct aperture photomtery on a FITS image. The user inputs a FITS image and a position of a star inputed as a ra/dec, pixels or through clicking on the star using an interactive window in matplotlib.

In [ ]:
#start with importing relevant libraries 

import numpy as np
import matplotlib
matplotlib.use('TkAgg') #interactive backend
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm #allows us to use log scale for colors

from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.visualization import astropy_mpl_style
import astropy.units as u
from photutils.detection import DAOStarFinder
from astropy.stats import sigma_clipped_stats
from astropy.wcs import WCS
from photutils.aperture import CircularAperture 
from photutils.aperture import CircularAnnulus
from photutils.aperture import ApertureStats


import pandas as pd
import os
import sys

In [ ]:
import warnings
from astropy.wcs import FITSFixedWarning

warnings.simplefilter('ignore', FITSFixedWarning)

## Reading the FITS image

The function below is designed to take an input of a FITS image and return the image's data and header. 

In [ ]:
def show_image (image_name_FITS):
    
    """This is a function that will take a FITS image, convert it to data 
    about the brightness of each pixel and plot it using matplotlib
    input:
    image_name_FITS = fits image provided by the user
    returns:
    image_FITS_data 
    header"""

    #retrive data and open 
    image_FITS = fits.open (image_name_FITS)
    image_FITS_data = image_FITS[0].data
    header = image_FITS[0].header

    
    #plot the FITS image
    plt.figure ()
    plt.imshow(image_FITS_data, origin = 'lower', norm = LogNorm(), cmap = 'Greys')
    plt.colorbar ()
    plt.show()
    return image_FITS_data, header

We can test this function below, by picking a FITS image to open. If the function runs sucessfully, it should open the FITS image in an interactive pop-up window:

In [ ]:
#try this out (note that the image has to be in the same folder, or extra path specification is required)
data, header = show_image("./01_LHS_33.fits")

## Aperture photometry

The user can now use the function "aperture_photomtery" to conduct aperture photomtery on a star from the loaded FITS image. The function takes input of the position of the target and the calibartion constant used. The possible inputs for the targets position are:
- Clicking on the target in the interactive pop up window (input: "click")
- Target's pixel position (input: A tuple with the x and y coordinates)
- Target's equatorial position (input: A tuple with RA and Dec in hours, mins, seconds format)

Below we state an example for each input method.

In [ ]:
def aperture_photometry (position, calibration): 
    """This function uses def show_image and the input of the user's coordinate of a 
     stars location and conducts aperture photometry
     input:
     position = clicking on source in interactive window / pixel coords / equatorial coords
     calibration = calibration constant used for computing magnitude
     returns:
     A Pandas dataframe saved as a CSV file containing the target's id, position (pixel and equatorial coords),
     sum, median sky background, npix, flux and magnitude """

    wcs = WCS(header)
    
    #locating stars in image using DAOStarFinder
    mean, median, std = sigma_clipped_stats(data, sigma = 3)
    print (f"For the provided FITS image the mean is {mean}, median is {median} and the standard deviation is {std}.")

    targetid = None
    fwhm_source = 2.5
    daofind = DAOStarFinder (fwhm = fwhm_source, threshold = 5.0*std) #threshold gives how much above background level

    #make table to store sources 
    sources = daofind (data-median)
    for col in sources.colnames:
        if col not in ('id', 'npix'):
            sources[col].info.format = '%.2f'

    #position input as pixels in tuple format
    if isinstance (position, tuple) and all(isinstance(p, (int, float)) for p in position):
        x_target, y_target = position
        print ("Confirmation that the coords are:", x_target, y_target )

    #input as RA & DEC
    elif isinstance(position, tuple) and len(position) == 2 and isinstance(position[0], str) and isinstance(position[1], str):
        coord = SkyCoord(position[0], position[1], unit=(u.hourangle, u.deg),frame='icrs')
        wcs = WCS(header)
        x_target, y_target = wcs.world_to_pixel(coord)
        print("RA/Dec resolved to pixel coords:", x_target, y_target)
  
    #position input through interactive backend
    elif isinstance (position, str): 
        if position == "click":
            plt.imshow(data, origin='lower', norm = LogNorm(), cmap = 'Greys')
            plt.title("Click on target star")
            coords = plt.ginput(1)
            plt.close()
            x_target, y_target = coords[0]
            print ("The coords found are:", x_target, y_target)
        else:
            print ("Incorrect string input. Try again.")
            sys.exit()
               
    else:
        print ("Incorrect coordinate format. Try again.")
        sys.exit()
    
    
    #matching to star
    if sources is None:
        raise ValueError("No sources detected. Try again.")
        sys.exit()

    dx = sources['xcentroid'] - x_target
    dy = sources['ycentroid'] - y_target
    dist = np.sqrt(dx**2 + dy**2)

    idx = np.argmin(dist)
    target = sources[idx]

    if dist[idx] > 10:
        print("Warning: No star found near target position. Try again.")
        sys.exit()
    else:
        print("Matched star:", target)
        targetid = idx+1
    
    #perform apperture photometry
    positions = np.transpose ((target['xcentroid'], target ['ycentroid']))
    apertures = CircularAperture(positions, r = 2 * fwhm_source) #r = radius of aperture, appropriate is 1.5 but here we match the radius for the DS9 task
    annulus_aperture = CircularAnnulus(positions, r_in = 4 * fwhm_source, r_out = 5 * fwhm_source) # define annulus; r_in & r_out need definition for number of pixels

    #plot
    plt.figure ()
    plt.imshow(data, cmap = 'Greys', norm = LogNorm(), origin = 'lower')
    apertures.plot (color = 'blue', lw = 1.5, alpha = 0.5);
    annulus_aperture.plot (color = 'green', lw = 1.5, alpha = 0.5);
    plt.show()

    
    #the stats
    aperstats = ApertureStats(data, apertures)
    anstats = ApertureStats (data, annulus_aperture)

    #finding values 
    sum = aperstats.sum
    skymedian = anstats.median
    npix = apertures.area

    #calculating flux
    flux = sum - (npix * skymedian)

    #calculating magnitude 
    mag = -2.5*np.log10(flux) + calibration

    #converting pixel coords to equatorial for output data frame
    coords = x_target, y_target
    ra_deg, dec_deg = wcs.pixel_to_world(x_target, y_target).ra.deg, \
                  wcs.pixel_to_world(x_target, y_target).dec.deg


    #define a dictionary and then store as pandas dataframe
    results = { "Image Name": image_name_FITS, "Target": targetid, "Pixel Coords": coords, "RA (degrees)": round(ra_deg, 5), 
               "Dec (Degrees)": round(dec_deg, 5), "Sum (S)": round(sum, 3), "Sky Median (B)": round(skymedian,3), 
               "npix": round(npix, 3), "Flux": round(flux,3) , "Magnitude": round(mag,3)}
    
    dataframe = pd.DataFrame([results])

    #export dataframe | only adds a new line for each new source
    filename = "aperture_photometry_results.csv"
    dataframe_new = pd.DataFrame([results])
    if os.path.exists(filename):
        dataframe = pd.read_csv(filename)
    
        duplicate = ((dataframe["Image Name"] == image_name_FITS) & (dataframe["Target"] == targetid) & np.isclose(dataframe["Sum (S)"], sum))
    
        if duplicate.any():
            print("Result already exists — not saving duplicate.")
        else:
            dataframe_new.to_csv(filename, mode='a', header=False, index=False)
    else:
        dataframe_new.to_csv(filename, index=False)

    return 

Here are some examples for calling the function with different position: 

**A) Clicking on the star through the interactive window:**

In [ ]:
aperture_photometry("click", 22.757)

**B) Pixel coordinates:**

In [ ]:
aperture_photometry((753,593), 22.757)

**C) Equatorial coordinates (RA and Dec):**

In [ ]:
aperture_photometry(("07h27m25.03s", "+05d12m30.0s"), 22.757)

## Conducting aperture photometry

Call the two functions obtain a CSV file containing information on the targets:

In [ ]:
#load the FITS image
data, header = show_image("./")

In [ ]:
#specify the position and calibration constant
aperture_photometry(,)

**A note on the output:** As mentioned beforehand, the data is saved in a Pandas dataframe that is outputed as a CSV file. Please note that the function is designed so that information on the chosen target will be added to the data frame only if it does not contain it yet. If it does, the output will simply say "Result already exists — not saving duplicate."  This saves makes the output more legible for the user. Additionally, the user can also conduct aperture photometry on multiple images and their sources. As the image name is saved in the pd data frame, the output will still be sensible. Happy coding!